# Agentic AI Systems — Worklet 13
# Research Grant Management System (apniLeap AURA)
## Lab 5 · MCP-Style Connector — One Server for All Sources

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** Expose every data source through one typed connector

---

**Description**

A Model Context Protocol (MCP) inspired integration server that acts as the single
governed surface for all six grant data sources:
1. `funding`: Verified agency calls and guideline text.
2. `literature`: Approved academic papers and citation records.
3. `history`: Prior institutional grant submissions and outcomes.
4. `hr`: Staff salary register and institutional rate card.
5. `policy`: Institutional rules (ethics, overhead, AI disclosure).
6. `rules`: Agency compliance rulebooks and document registries.

**Key concepts demonstrated**

- Unified capability manifest with typed input/output contracts.
- Distinction between `resource` (static data) and `tool` (dynamic query/compute).
- Authentication boundary: internal agent token required for all data access.
- Authorization: Caller allowlisting for sensitive endpoints (e.g., HR salary records).
- Capability discovery: Client dynamically discovers and invokes endpoints without hardcoded methods.
- Full audit logging of all data reads.

**Assignments included**

- Assignment 1: Dynamic capability discovery (adding a fourth capability 'history.stats' discovered at runtime).
- Assignment 2: Security boundary check (enforcing token check and caller restrictions).

---

*Runs standalone on Python 3.10+ (Jupyter or Google Colab): no API key, no network, standard library only. Source script: [`scripts/lab05_mcp_connector.py`](./scripts/lab05_mcp_connector.py).*

### Cell 1 · Shared institutional seed data

`seed_data.py` is written to disk so the notebook runs standalone. It is the Python mirror of the course project's `backend/src/seed.js`: funding calls, literature corpus, policies, personnel rate card, users and the sanctioned-project portfolio. *(Collapse — shared by all labs.)*

In [1]:
%%writefile seed_data.py
"""
AURA Seed Data & Corpus -- Research Grant Management System
Corresponds to backend/src/seed.js, faculty.js, and projects.js
"""

INSTITUTION = {
    "id": "inst-mist",
    "name": "Meridian Institute of Science and Technology",
    "short": "MIST",
    "type": "university",
    "city": "Pune",
    "country": "IN",
    "researchOffice": "Research & Development Office",
    "signatory": "Dean, Research & Development",
}

USERS = [
    {
        "id": "usr-krishnan",
        "name": "Dr. Uma Mudengudi",
        "email": "uma.mudengudi@mist.edu.in",
        "role": "dean",
        "title": "Dean, Research & Development",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback", "metrics:read"],
    },
    {
        "id": "usr-suneeta-v-budihal",
        "name": "Dr. Suneeta V Budihal",
        "email": "suneeta.v.budihal@mist.edu.in",
        "role": "pi",
        "title": "Associate Professor",
        "department": "SoECE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-satyadhyan-chickerur",
        "name": "Dr. Satyadhyan Chickerur",
        "email": "satyadhyan.chickerur@mist.edu.in",
        "role": "pi",
        "title": "Professor & Head",
        "department": "CSE",
        "permissions": ["run:create", "run:read:own", "draft:edit", "package:view"],
    },
    {
        "id": "usr-research-office",
        "name": "Research & Development Office",
        "email": "grants.cell@mist.edu.in",
        "role": "research_office",
        "title": "Grants Cell & Compliance",
        "department": "Office of Dean R&D",
        "permissions": ["run:create", "run:read:all", "review:approve", "package:view", "audit:read", "version:rollback"],
    },
]

RATE_CARD = {
    "institutionId": "inst-mist",
    "overheadPct": 15,
    "contingencyPct": 5,
    "salaryEscalationPct": 5,
    "equipmentMaintenancePct": 4,
    "effectiveFrom": "2026-04-01",
    "note": "Institutional default rates. Agency-published rates for a call always take precedence.",
}

PERSONNEL = [
    {"id": "per-uma", "name": "Dr. Uma Mudengudi", "role": "PI", "title": "Dean (R&D) & Professor", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "professor"},
    {"id": "per-suneeta", "name": "Dr. Suneeta V Budihal", "role": "PI", "title": "Faculty Member", "department": "SoECE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-satyadhyan", "name": "Dr. Satyadhyan Chickerur", "role": "PI", "title": "Faculty Member", "department": "CSE", "monthlyCost": 0, "defaultEffortPct": 10, "careerStage": "associate"},
    {"id": "per-ra", "name": "Research Associate (to be appointed)", "role": "RA", "title": "Research Associate", "department": "Research Office", "monthlyCost": 48000, "defaultEffortPct": 100, "careerStage": "n/a"},
    {"id": "per-phd", "name": "Doctoral Scholar (to be appointed)", "role": "PhD", "title": "Doctoral Scholar", "department": "Research Office", "monthlyCost": 36000, "defaultEffortPct": 100, "careerStage": "n/a"},
]

POLICIES = [
    {
        "id": "pol-overhead",
        "title": "Institutional overhead and costing policy",
        "tags": ["budget", "overhead"],
        "text": "Overheads are charged at the rate published by the funding agency for the call. Where the agency publishes no rate, the institutional default of 15% of total direct costs applies. Overhead may not fund permanent faculty salaries.",
    },
    {
        "id": "pol-ethics",
        "title": "Human subjects research ethics policy",
        "tags": ["compliance", "ethics", "health"],
        "text": "Any proposal involving human participants, identifiable health data or clinical validation requires Institutional Ethics Committee approval before the award is activated. A conditional certificate may be attached at submission and must be converted to full approval before funds are released.",
    },
    {
        "id": "pol-data",
        "title": "Research data protection policy (DPDP-aligned)",
        "tags": ["compliance", "data", "privacy"],
        "text": "Proposals collecting personal data must include a Data Management Plan covering lawful basis, de-identification, storage location, retention period and a named data steward. Cross-border transfer of health data requires a documented risk assessment.",
    },
    {
        "id": "pol-ai",
        "title": "Disclosure of AI-assisted writing",
        "tags": ["compliance", "ai", "authorship"],
        "text": "AI-assisted drafting must be disclosed in the proposal cover declaration and in the institutional submission record. AI-generated text is a draft until a named faculty member reviews and accepts it; AI tools may not be listed as authors or investigators.",
    },
    {
        "id": "pol-coi",
        "title": "Conflict of interest declaration",
        "tags": ["compliance", "coi"],
        "text": "Every investigator must declare financial interests, advisory roles and institutional relationships that could reasonably be perceived to influence the research. Declarations are collected by the Research Office and retained for the audit period.",
    },
    {
        "id": "pol-signoff",
        "title": "Authorised signatory policy",
        "tags": ["compliance", "approval"],
        "text": "Only the Dean, Research & Development, or a delegated Research Office signatory, may authorise submission to a funding agency. The authorising officer must record the decision, the version reviewed, and any conditions attached to approval.",
    },
]

COMPLIANCE_RULES = [
    {"id": "rule-sections", "kind": "sections", "appliesTo": "all", "severity": "blocking", "requirement": "Every section required by the call is present and non-empty.", "evidence": "Call required-section list vs draft section index."},
    {"id": "rule-pages", "kind": "pageLimit", "appliesTo": "all", "severity": "blocking", "requirement": "The narrative stays within the page limit published for the call.", "evidence": "Word count converted at 500 words per page."},
    {"id": "rule-budget-ceiling", "kind": "budgetCeiling", "appliesTo": "all", "severity": "blocking", "requirement": "The requested total does not exceed the agency ceiling for this call.", "evidence": "Budget total vs call award maximum."},
    {"id": "rule-equipment-cap", "kind": "equipmentCap", "appliesTo": "all", "severity": "blocking", "requirement": "Equipment does not exceed the percentage ceiling in the call guidelines.", "evidence": "Equipment subtotal / total requested."},
    {"id": "rule-salary-cap", "kind": "salaryCap", "appliesTo": "all", "severity": "blocking", "requirement": "Manpower costs stay within the call salary ceiling.", "evidence": "Personnel subtotal / total requested."},
    {"id": "rule-overhead", "kind": "overheadRate", "appliesTo": "all", "severity": "blocking", "requirement": "Overhead is charged at exactly the rate published for the call.", "evidence": "Overhead amount vs rate x direct costs."},
    {"id": "rule-ethics", "kind": "document", "docKey": "ethics", "appliesTo": "all", "severity": "blocking", "requirement": "Ethics clearance is attached where the call or the research involves human participants.", "evidence": "Document register vs call requirement."},
    {"id": "rule-coi", "kind": "document", "docKey": "coi", "appliesTo": "all", "severity": "blocking", "requirement": "Conflict-of-interest declarations are attached for all investigators.", "evidence": "Document register vs call requirement."},
    {"id": "rule-dmp", "kind": "document", "docKey": "dmp", "appliesTo": "all", "severity": "blocking", "requirement": "A Data Management Plan is attached where required.", "evidence": "Document register vs call requirement."},
    {"id": "rule-disclosure", "kind": "aiDisclosure", "appliesTo": "all", "severity": "blocking", "requirement": "AI-assisted drafting is disclosed in the submission declaration.", "evidence": "Institutional AI disclosure policy vs package declaration."},
]

DOCUMENT_REGISTER = [
    {"key": "ethics", "label": "Institutional Ethics Committee clearance", "status": "conditional", "note": "Conditional certificate available; full approval pending protocol review."},
    {"key": "coi", "label": "Conflict-of-interest declarations (all investigators)", "status": "attached", "note": "Signed by all investigators."},
    {"key": "dmp", "label": "Data Management Plan", "status": "attached", "note": "Deposited in the institutional DMP registry."},
    {"key": "budget-justification", "label": "Budget justification", "status": "attached", "note": "Generated by the Budget Planning Agent."},
    {"key": "csr-1", "label": "CSR-1 registration and company CSR policy", "status": "pending", "note": "Company secretarial certificate requested."},
    {"key": "consortium-agreement", "label": "Consortium agreement and partner letters", "status": "pending", "note": "Letters of intent collected; agreement under legal review."},
]

CALLS = [
    {
        "id": "call-serb-crg",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "Core Research Grant (CRG)",
        "title": "SERB Core Research Grant 2026 -- Individual Investigator",
        "fundingType": "government",
        "domains": ["ai-ml", "energy", "materials", "health", "climate"],
        "tags": ["artificial intelligence", "machine learning", "federated learning", "sustainable energy", "materials science", "medical imaging"],
        "deadline": "2026-12-15",
        "budgetMin": 3000000,
        "budgetMax": 7500000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-serb-crg#g1", "section": "submission", "text": "Proposals must be routed through the institutional research office; individual submissions are not accepted."},
            {"id": "call-serb-crg#g2", "section": "budget", "text": "The total request may not exceed INR 75,00,000 for a duration of up to 36 months."},
            {"id": "call-serb-crg#g3", "section": "budget", "text": "Equipment may not exceed 40% of the total requested budget; requests above this ceiling require a written justification from the Head of Institution."},
            {"id": "call-serb-crg#g4", "section": "budget", "text": "Overheads are payable at 15% of total direct costs."},
            {"id": "call-serb-crg#g5", "section": "documents", "text": "A Data Management Plan and a conflict-of-interest declaration are mandatory at submission."},
            {"id": "call-serb-crg#g6", "section": "format", "text": "The narrative may not exceed 25 pages excluding references and the budget annexure."},
        ],
    },
    {
        "id": "call-icmr-adhoc",
        "agency": "ICMR",
        "agencyFull": "Indian Council of Medical Research",
        "program": "Ad-hoc Research Project",
        "title": "ICMR Ad-hoc Research Project 2026 -- Diagnostics & Public Health",
        "fundingType": "government",
        "domains": ["health", "ai-ml", "public-health"],
        "tags": ["diabetic retinopathy", "medical imaging", "rural health", "screening", "federated learning", "digital health", "primary care"],
        "deadline": "2026-12-05",
        "budgetMin": 2000000,
        "budgetMax": 6000000,
        "durationMonths": 36,
        "pageLimit": 25,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 50, "equipmentCapPct": 30, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [
            {"id": "call-icmr-adhoc#g1", "section": "eligibility", "text": "Applicants must hold a regular position in a recognised institution or hospital and demonstrate access to the study population."},
            {"id": "call-icmr-adhoc#g2", "section": "budget", "text": "The total request must fall between INR 20,00,000 and INR 60,00,000; equipment is capped at 30%."},
            {"id": "call-icmr-adhoc#g3", "section": "ethics", "text": "Institutional Ethics Committee approval is mandatory for any study involving human participants or their data."},
            {"id": "call-icmr-adhoc#g4", "section": "data", "text": "A Data Management Plan describing de-identification and retention is required; identifiable health data may not leave the institution without a documented risk assessment."},
            {"id": "call-icmr-adhoc#g5", "section": "deliverables", "text": "The proposal must state measurable clinical or public-health outcomes for the funding period."},
        ],
    },
    {
        "id": "call-tata-csr",
        "agency": "Tata Trusts",
        "agencyFull": "Tata Trusts -- Corporate Social Responsibility",
        "program": "CSR Research Partnership",
        "title": "Tata Trusts CSR Research Partnership 2026 -- Health & Education",
        "fundingType": "industry",
        "domains": ["health", "education", "public-health"],
        "tags": ["community health", "digital health", "education technology", "rural outreach", "screening"],
        "deadline": "2026-11-15",
        "budgetMin": 2500000,
        "budgetMax": 10000000,
        "durationMonths": 24,
        "pageLimit": 20,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital", "ngo"], "consortiumRequired": False, "minPartners": 0, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 8, "salaryCapPct": 55, "equipmentCapPct": 25, "travelCapPct": 10, "contingencyPct": 3},
        "guidelines": [
            {"id": "call-tata-csr#g1", "section": "compliance", "text": "The activity must fall within Schedule VII of the Companies Act 2013 and the implementing partner must hold a valid CSR-1 registration."},
            {"id": "call-tata-csr#g2", "section": "budget", "text": "Grants range from INR 25,00,00,000 to INR 1,00,00,000 over 24 months; overheads are capped at 8%."},
            {"id": "call-tata-csr#g3", "section": "reporting", "text": "Quarterly outcome reporting against agreed indicators is a condition of release of the second tranche."},
        ],
    },
    {
        "id": "call-eu-horizon",
        "agency": "EU Horizon Europe",
        "agencyFull": "European Commission -- Horizon Europe",
        "program": "Cluster 4: Digital, Industry and Space",
        "title": "Horizon Europe 2027 -- Trustworthy AI for Health Systems (RIA)",
        "fundingType": "international",
        "domains": ["ai-ml", "health", "digital"],
        "tags": ["trustworthy ai", "federated learning", "health systems", "digital health", "data governance", "open science"],
        "deadline": "2027-01-20",
        "budgetMin": 18000000,
        "budgetMax": 45000000,
        "durationMonths": 36,
        "pageLimit": 45,
        "verified": True,
        "status": "open",
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university", "institute", "hospital"], "consortiumRequired": True, "minPartners": 3, "coFundingRequired": False},
        "budgetRules": {"overheadPct": 25, "salaryCapPct": 40, "equipmentCapPct": 30, "travelCapPct": 15, "contingencyPct": 0},
        "guidelines": [
            {"id": "call-eu-horizon#g1", "section": "eligibility", "text": "Proposals require a consortium of at least three independent legal entities from three different countries."},
            {"id": "call-eu-horizon#g2", "section": "budget", "text": "Personnel costs may not exceed 40% and equipment 30% of the total eligible cost; a 25% flat rate covers indirect costs."},
            {"id": "call-eu-horizon#g3", "section": "open-science", "text": "Open science practices, FAIR data management and a gender and inclusion plan are evaluated under the impact criterion."},
        ],
    },
    {
        "id": "call-serb-supra-expired",
        "agency": "SERB",
        "agencyFull": "Science and Engineering Research Board, DST",
        "program": "SUPRA",
        "title": "SERB SUPRA 2026 -- Closed",
        "fundingType": "government",
        "domains": ["ai-ml", "health"],
        "tags": ["artificial intelligence", "medical imaging"],
        "deadline": "2026-06-30",
        "budgetMin": 1000000,
        "budgetMax": 3000000,
        "durationMonths": 24,
        "status": "closed",
        "verified": True,
        "eligibility": {"careerStages": ["assistant", "associate", "professor"], "institutionTypes": ["university"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 15, "salaryCapPct": 45, "equipmentCapPct": 40, "travelCapPct": 10, "contingencyPct": 5},
        "guidelines": [{"id": "call-serb-supra-expired#g1", "section": "status", "text": "This call closed on 30 June 2026 and is retained for reference only."}],
    },
    {
        "id": "call-unverified-sample",
        "agency": "Unknown Aggregator",
        "agencyFull": "Third-party opportunity aggregator",
        "program": "Sample Aggregator Call",
        "title": "Unverified Grant Opportunity 2027",
        "fundingType": "government",
        "domains": ["engineering"],
        "tags": ["engineering"],
        "deadline": "2027-02-01",
        "budgetMin": 800000,
        "budgetMax": 2500000,
        "durationMonths": 24,
        "verified": False,
        "status": "unverified",
        "eligibility": {"careerStages": ["assistant"], "institutionTypes": ["college"], "consortiumRequired": False, "minPartners": 0},
        "budgetRules": {"overheadPct": 10, "salaryCapPct": 40, "equipmentCapPct": 35, "travelCapPct": 12, "contingencyPct": 5},
        "guidelines": [{"id": "call-unverified#g1", "section": "status", "text": "Listing could not be confirmed against an official agency notification."}],
    },
]

LITERATURE = [
    {
        "id": "lit-001",
        "title": "Federated learning for diabetic retinopathy screening across rural clinics",
        "authors": ["S. Banerjee", "R. Iyer", "M. Kulkarni"],
        "year": 2024,
        "venue": "Journal of Medical AI Research",
        "tags": ["federated learning", "diabetic retinopathy", "rural health", "medical imaging", "screening"],
        "abstract": "A four-site federated study reports comparable sensitivity to centralised training while keeping 1.2 million fundus images inside participating clinics.",
        "citations": 61,
        "doi": "10.5555/aura.demo.001",
    },
    {
        "id": "lit-002",
        "title": "Deep learning detection of referable diabetic retinopathy in primary care",
        "authors": ["L. Fernandes", "A. Gupta"],
        "year": 2023,
        "venue": "Ophthalmology Informatics",
        "tags": ["diabetic retinopathy", "medical imaging", "primary care", "screening", "deep learning"],
        "abstract": "A prospective primary-care evaluation of 8,400 patients reports 92% sensitivity and 88% specificity for referable retinopathy.",
        "citations": 148,
        "doi": "10.5555/aura.demo.002",
    },
    {
        "id": "lit-003",
        "title": "Privacy-preserving federated learning in healthcare: a systematic review",
        "authors": ["H. Park", "J. Okafor", "N. Singh"],
        "year": 2024,
        "venue": "Health Data Science Review",
        "tags": ["federated learning", "privacy", "health systems", "data governance"],
        "abstract": "Reviews 74 studies and finds that differential privacy and secure aggregation are rarely combined in clinical deployments.",
        "citations": 96,
        "doi": "10.5555/aura.demo.003",
    },
    {
        "id": "lit-004",
        "title": "Non-mydriatic fundus photography in resource-limited settings",
        "authors": ["P. Adjei", "T. Nguyen"],
        "year": 2022,
        "venue": "Global Eye Health",
        "tags": ["rural health", "screening", "medical imaging", "primary care"],
        "abstract": "Image quality is the dominant failure mode for non-mydriatic cameras operated by trained community health workers.",
        "citations": 57,
        "doi": "10.5555/aura.demo.004",
    },
    {
        "id": "lit-005",
        "title": "Teleophthalmology screening pathways in low-resource districts",
        "authors": ["K. Rao", "D. Bhattacharya"],
        "year": 2023,
        "venue": "Indian Journal of Public Health",
        "tags": ["screening", "rural health", "health systems", "telemedicine"],
        "abstract": "Hub-and-spoke teleophthalmology reduced time-to-diagnosis from 26 to 9 days across six districts.",
        "citations": 73,
        "doi": "10.5555/aura.demo.005",
    },
]

HISTORY = [
    {
        "id": "hist-retina-2024",
        "title": "AI-assisted retinal screening for diabetic retinopathy in Pune district",
        "agency": "ICMR",
        "year": 2024,
        "outcome": "not-funded",
        "amount": 5600000,
        "tags": ["diabetic retinopathy", "screening", "rural health", "medical imaging"],
        "abstract": "Proposed a three-site pilot of automated retinal grading with referral support for 12,000 patients.",
        "lessons": "Reviewers questioned the missing ethics clearance and the absence of a site-level data-governance plan. Both are now prerequisites.",
    },
    {
        "id": "hist-fl-testbed-2023",
        "title": "Federated learning testbed for multi-hospital medical imaging",
        "agency": "SERB",
        "year": 2023,
        "outcome": "funded",
        "amount": 6200000,
        "tags": ["federated learning", "medical imaging", "privacy", "testbed"],
        "abstract": "Built a three-hospital federated learning testbed with secure aggregation and site-level audit logging.",
        "lessons": "Delivered the testbed; clinical validation was out of scope and remains the open gap.",
    },
]


Writing seed_data.py


### Cell 2 · Imports and fixtures

In [2]:
import time
from typing import Dict, Any, List, Optional
from seed_data import (
    CALLS, LITERATURE, HISTORY, PERSONNEL, RATE_CARD, POLICIES,
    COMPLIANCE_RULES, DOCUMENT_REGISTER, INSTITUTION
)

CONNECTOR_TOKEN = "aura-internal-agent-token"

# Manifest declaring all exposed resources and tools
CAPABILITIES = [
    {
        "name": "funding.calls",
        "kind": "resource",
        "source": "funding",
        "description": "Lists verified funding calls filtered by funding type",
        "input": {"fundingType": "Optional[str]"},
        "output": {"calls": "List[dict]"},
    },
    {
        "name": "hr.personnel",
        "kind": "resource",
        "source": "hr",
        "description": "Reads personnel register and monthly salaries",
        "input": {"role": "Optional[str]"},
        "output": {"personnel": "List[dict]"},
    },
    {
        "name": "compliance.rules",
        "kind": "resource",
        "source": "rules",
        "description": "Reads compliance checklist rules for a given funding type",
        "input": {"appliesTo": "Optional[str]"},
        "output": {"rules": "List[dict]"},
    },
    {
        "name": "query_guidelines",
        "kind": "tool",
        "source": "funding",
        "description": "Queries specific guidelines for a call with section filter",
        "input": {"callId": "str", "section": "Optional[str]"},
        "output": {"guidelines": "List[dict]"},
    },
]

ALLOWED_CALLERS_HR = {"coordinator", "budget_agent", "dean"}

### Cell 3 · `MCPServer` (class)

In [3]:
class MCPServer:
    """
    Central Integration Server fronting all grant sources.
    No agent is permitted to touch a raw source directly.
    """
    def __init__(self):
        self.audit_log: List[Dict[str, Any]] = []

    def list_capabilities(self) -> List[Dict[str, Any]]:
        """Capability discovery endpoint."""
        return CAPABILITIES

    def invoke(self, endpoint: str, args: Dict[str, Any], token: str, caller: str) -> Dict[str, Any]:
        """Governed execution of an MCP capability."""
        start_time = time.time()
        base_log = {
            "endpoint": endpoint,
            "caller": caller,
            "args": args,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        }

        # 1. Authentication Check
        if token != CONNECTOR_TOKEN:
            err = {"error": "Unauthorized: invalid agent token"}
            self._log_audit({**base_log, "status": "error", "error": err["error"]})
            return err

        # 2. Authorization Check for Sensitive HR Data
        if endpoint == "hr.personnel" and caller not in ALLOWED_CALLERS_HR:
            err = {"error": f"Permission denied: caller '{caller}' cannot access HR personnel records"}
            self._log_audit({**base_log, "status": "denied", "error": err["error"]})
            return err

        # 3. Dispatch to internal handlers
        try:
            if endpoint == "funding.calls":
                ft = args.get("fundingType")
                data = [c for c in CALLS if not ft or c.get("fundingType") == ft]
                res = {"calls": data, "count": len(data)}

            elif endpoint == "hr.personnel":
                role = args.get("role")
                data = [p for p in PERSONNEL if not role or p.get("role") == role]
                res = {"personnel": data, "count": len(data)}

            elif endpoint == "compliance.rules":
                app = args.get("appliesTo")
                data = [r for r in COMPLIANCE_RULES if not app or r.get("appliesTo") in ("all", app)]
                res = {"rules": data, "count": len(data)}

            elif endpoint == "query_guidelines":
                call_id = args.get("callId")
                section = args.get("section")
                call = next((c for c in CALLS if c["id"] == call_id), None)
                if not call:
                    return {"error": f"Call '{call_id}' not found"}
                guides = call.get("guidelines", [])
                if section:
                    guides = [g for g in guides if g.get("section") == section]
                res = {"guidelines": guides, "count": len(guides)}

            elif endpoint == "history.stats":
                # Added for Assignment 1
                total = len(HISTORY)
                funded = sum(1 for h in HISTORY if h.get("outcome") == "funded")
                res = {"total": total, "funded": funded, "success_rate": round(funded / total, 2)}

            else:
                return {"error": f"Unknown capability '{endpoint}'"}

            latency_ms = round((time.time() - start_time) * 1000, 2)
            self._log_audit({**base_log, "status": "ok", "latency_ms": latency_ms, "count": res.get("count", 1)})
            return {"result": res, "latency_ms": latency_ms}

        except Exception as e:
            return {"error": f"Server invocation error: {str(e)}"}

    def _log_audit(self, entry: Dict[str, Any]):
        self.audit_log.append(entry)

### Cell 4 · `MCPClient` (class)

In [4]:
class MCPClient:
    """
    Client used by all pipeline agents to discover and invoke capabilities.
    """
    def __init__(self, server: MCPServer, caller_name: str, token: str = CONNECTOR_TOKEN):
        self.server = server
        self.caller_name = caller_name
        self.token = token

    def discover(self) -> List[Dict[str, Any]]:
        """Dynamically queries the server's manifest."""
        return self.server.list_capabilities()

    def call(self, capability_name: str, **kwargs) -> Dict[str, Any]:
        """Dispatches dynamic call through the discovered interface."""
        manifest = {c["name"]: c for c in self.discover()}
        if capability_name not in manifest:
            raise ValueError(f"Capability '{capability_name}' not available in server manifest")

        return self.server.invoke(
            endpoint=capability_name,
            args=kwargs,
            token=self.token,
            caller=self.caller_name,
        )

### Cell 5 · `run_assignment_1_add_capability` (function)

In [5]:
# ==============================================================================
# Graded Assignments
# ==============================================================================

def run_assignment_1_add_capability(server: MCPServer):
    """
    Assignment 1 -- Add a Fourth, Discoverable Capability:
    Add 'history.stats' to CAPABILITIES and verify that MCPClient can
    discover and invoke it without any hard-coded methods added to MCPClient.
    """
    print("\n--- [Assignment 1] Dynamic Discovery of New Capability ---")
    CAPABILITIES.append({
        "name": "history.stats",
        "kind": "resource",
        "source": "history",
        "description": "Aggregates historical grant success rates",
        "input": {},
        "output": {"total": "int", "funded": "int", "success_rate": "float"},
    })

    client = MCPClient(server, caller_name="coordinator")
    discovered = [c["name"] for c in client.discover()]
    print(f"Discovered Capabilities: {discovered}")
    assert "history.stats" in discovered, "New capability was not discovered!"

    stats = client.call("history.stats")
    print(f"Invoked 'history.stats' dynamically: {stats['result']}")

### Cell 6 · `run_assignment_2_security_boundary` (function)

In [6]:
def run_assignment_2_security_boundary(server: MCPServer):
    """
    Assignment 2 -- Security Boundary Enforcement:
    1. Test invalid token rejection.
    2. Test unauthorized caller accessing restricted HR records.
    """
    print("\n--- [Assignment 2] Security Boundary Enforcement ---")

    # Adversarial test 1: Bad token
    unauthorized_client = MCPClient(server, caller_name="coordinator", token="invalid-secret-key")
    res1 = unauthorized_client.call("funding.calls", fundingType="government")
    print(f"Bad Token Test: {res1}")
    assert "error" in res1 and "Unauthorized" in res1["error"], "Token security check bypassed!"

    # Adversarial test 2: Unpermitted caller accessing HR
    guest_client = MCPClient(server, caller_name="guest_agent", token=CONNECTOR_TOKEN)
    res2 = guest_client.call("hr.personnel")
    print(f"Unpermitted Caller Test: {res2}")
    assert "error" in res2 and "Permission denied" in res2["error"], "Caller permission check bypassed!"
    print("Verification passed: Security boundaries strictly enforced.")

### Cell 7 · Run the lab

The lab demo and both graded assignments, with the console captured for the checks below.

In [7]:
# Console capture: everything the lab prints is shown below AND kept in memory,
# so the acceptance checks in the last cell can assert on the real run.
import io
import sys


class ConsoleTee(io.StringIO):
    """Mirror stdout into a buffer while the lab runs."""

    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, *exc):
        sys.stdout = self._stdout
        return False

    def write(self, text):
        self._stdout.write(text)
        return super().write(text)


console = ConsoleTee()

with console:
    print("======================================================================")
    print("Lab 5: MCP-Style Connector -- Mini Integration Server")
    print("======================================================================")

    server = MCPServer()
    client = MCPClient(server, caller_name="budget_agent")

    # 1. Capability Discovery
    print("=== Capability Discovery from Manifest ===")
    for cap in client.discover():
        print(f"  [{cap['kind'].upper()}] {cap['name']} -> {cap['description']}")

    # 2. Call 1: Read funding calls (resource)
    print("\n=== Call 1: Read Funding Calls ===")
    calls_res = client.call("funding.calls", fundingType="government")
    print(f"Retrieved {calls_res['result']['count']} government calls (Latency: {calls_res['latency_ms']} ms)")

    # 3. Call 2: Read HR Personnel (resource with authorized caller)
    print("\n=== Call 2: Read HR Personnel (Authorized caller 'budget_agent') ===")
    hr_res = client.call("hr.personnel", role="RA")
    print(f"Personnel records: {hr_res['result']['personnel']}")

    # 4. Call 3: Query Guidelines (tool)
    print("\n=== Call 3: Query Guidelines ===")
    guide_res = client.call("query_guidelines", callId="call-serb-crg", section="budget")
    print(f"Budget Guidelines for SERB CRG: {guide_res['result']['guidelines']}")

    # 5. Assignments
    run_assignment_1_add_capability(server)
    run_assignment_2_security_boundary(server)

    # 6. Audit Log inspection
    print("\n=== Server Audit Log Sample ===")
    for log in server.audit_log[:4]:
        print(f"  {log['timestamp']} | Caller: {log['caller']} | Endpoint: {log['endpoint']} | Status: {log['status']}")

Lab 5: MCP-Style Connector -- Mini Integration Server
=== Capability Discovery from Manifest ===
  [RESOURCE] funding.calls -> Lists verified funding calls filtered by funding type
  [RESOURCE] hr.personnel -> Reads personnel register and monthly salaries
  [RESOURCE] compliance.rules -> Reads compliance checklist rules for a given funding type
  [TOOL] query_guidelines -> Queries specific guidelines for a call with section filter

=== Call 1: Read Funding Calls ===
Retrieved 4 government calls (Latency: 0.01 ms)

=== Call 2: Read HR Personnel (Authorized caller 'budget_agent') ===
Personnel records: [{'id': 'per-ra', 'name': 'Research Associate (to be appointed)', 'role': 'RA', 'title': 'Research Associate', 'department': 'Research Office', 'monthlyCost': 48000, 'defaultEffortPct': 100, 'careerStage': 'n/a'}]

=== Call 3: Query Guidelines ===
Budget Guidelines for SERB CRG: [{'id': 'call-serb-crg#g2', 'section': 'budget', 'text': 'The total request may not exceed INR 75,00,000 for a d

### Cell 8 · Acceptance checks and deliverables

Each check asserts on the trace of the run above. The trace and the check report are exported to `lab5_artifacts/` (committed under `Course-Project/evidence/`).

In [8]:
import json
import re
from pathlib import Path

LAB = 5
CHECKS = [('Capabilities are discovered from the manifest', '\\[TOOL\\] query_guidelines', 1),
    ('Funding calls are read through the connector', 'Retrieved \\d+ government calls', 1),
    ('A capability added later is discovered and invoked dynamically', "Invoked 'history\\.stats' dynamically", 1),
    ('A bad agent token is refused', 'Unauthorized: invalid agent token', 1),
    ('An unpermitted caller cannot read HR records', "Permission denied: caller 'guest_agent'", 1),
    ('Every connector call lands in the server audit log', '\\| Caller: \\w+ \\| Endpoint: [\\w.]+ \\| Status: ok', 4)]

trace = console.getvalue()
universal = [
    ("Run produced console output", r"\S", 1),
    ("Graded Assignment 1 executed", r"\[Assignment 1\]", 1),
    ("Graded Assignment 2 executed", r"\[Assignment 2\]", 1),
]
results = []
for label, pattern, minimum in universal + CHECKS:
    found = len(re.findall(pattern, trace, flags=re.MULTILINE))
    results.append({"check": label, "pattern": pattern, "expected_min": minimum,
                    "found": found, "passed": found >= minimum})
no_errors = "Traceback (most recent call last)" not in trace
results.append({"check": "No unhandled exception in the run", "pattern": "Traceback",
                "expected_min": 0, "found": 0 if no_errors else 1, "passed": no_errors})

for i, r in enumerate(results, 1):
    print(f"[{'PASS' if r['passed'] else 'FAIL'}] {i:>2}. {r['check']}")
passed = sum(r["passed"] for r in results)
print(f"\nLab {LAB}: {passed}/{len(results)} acceptance checks passed")

outdir = Path(f"lab{LAB}_artifacts")
outdir.mkdir(exist_ok=True)
(outdir / f"lab{LAB}_console_trace.txt").write_text(trace, encoding="utf-8")
(outdir / f"lab{LAB}_acceptance_checks.json").write_text(
    json.dumps({"lab": LAB, "passed": passed, "total": len(results), "checks": results}, indent=2),
    encoding="utf-8")
print("Artifacts written to", outdir.as_posix())

assert passed == len(results), "acceptance checks failed"

[PASS]  1. Run produced console output
[PASS]  2. Graded Assignment 1 executed
[PASS]  3. Graded Assignment 2 executed
[PASS]  4. Capabilities are discovered from the manifest
[PASS]  5. Funding calls are read through the connector
[PASS]  6. A capability added later is discovered and invoked dynamically
[PASS]  7. A bad agent token is refused
[PASS]  8. An unpermitted caller cannot read HR records
[PASS]  9. Every connector call lands in the server audit log
[PASS] 10. No unhandled exception in the run

Lab 5: 10/10 acceptance checks passed
Artifacts written to lab5_artifacts
